# Example 7 — SCF / misreconstructed-event migration

This notebook demonstrates Laura++-style self-cross-feed migration for $B^+\to K^+\pi^+\pi^-$ using $s_{13}$ and $s_{23}$.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BaBarFlatte, DecayChannel, DecayModel, LASS, NonResonant, RealImag,
    Resonance, enable_x64
)
enable_x64()

channel = DecayChannel("B+", ("K+", "pi+", "pi-"))
components = [
    Resonance("Kstar892", (0, 2), RealImag(1.0, 0.0), mass=0.8958, width=0.0474, spin=1, resonance_radius=4.0, parent_radius=4.0),
    Resonance("KpiS", (0, 2), RealImag(1.40, -0.60), lineshape=LASS(2.07, 3.32, 1.8), mass=1.425, width=0.270, spin=0, resonance_radius=4.0, parent_radius=4.0),
    Resonance("rho770", (1, 2), RealImag(0.65, 0.10), mass=0.7753, width=0.1491, spin=1, resonance_radius=4.0, parent_radius=4.0),
    Resonance("f0_980", (1, 2), RealImag(-0.20, 1.00), lineshape=BaBarFlatte(), mass=0.965, width=0.0, spin=0, resonance_radius=4.0, parent_radius=4.0),
    NonResonant(RealImag(-0.50, 0.10)),
]
model = DecayModel(channel, components, normalization_method="square-dalitz", normalization_resolution=300, normalization_pair=(0, 2))

from dalitzplotfitter import SCFSignalPDF, SquareDalitzSCFMap
from dalitzplotfitter.integration import GridIntegrator


## 1. Synthetic Square-Dalitz migration matrix

Rows are true bins and columns are reconstructed bins: $M_{ij}=P(j_{reco}|i_{true})$.


In [ ]:
N_MP = 20
N_TP = 20
n_bins = N_MP * N_TP
mp = (np.arange(N_MP) + 0.5) / N_MP
tp = (np.arange(N_TP) + 0.5) / N_TP
gmp, gtp = np.meshgrid(mp, tp, indexing="ij")
coords = np.column_stack((gmp.ravel(), gtp.ravel()))
migration = np.empty((n_bins, n_bins))
for i, (m0, t0) in enumerate(coords):
    d2 = ((coords[:,0]-m0)/0.045)**2 + ((coords[:,1]-t0)/0.055)**2
    row = np.exp(-0.5*d2)
    migration[i] = row / row.sum()
f_scf = 0.05 + 0.15*gmp.ravel()**2

scf_map = SquareDalitzSCFMap(
    migration=jnp.asarray(migration),
    scf_fraction=jnp.asarray(f_scf),
    mother_mass=channel.parent_mass,
    masses=channel.daughter_masses,
    bins_mprime=N_MP,
    bins_thetaprime=N_TP,
    pair=(0, 2),
)
print("max row-sum error:", np.max(np.abs(migration.sum(axis=1)-1.0)))
print("SCF fraction range:", f_scf.min(), f_scf.max())


## 2. SCF-aware signal PDF


In [ ]:
pdf = SCFSignalPDF(
    intensity=lambda data, pars: model.intensity(data, pars),
    integrator=GridIntegrator(model.normalization_sample),
    scf_map=scf_map,
)

true_data = scf_map.true_bin_data()
true_density = model.intensity(true_data)
areas = scf_map.phase_space_areas()
scf_density = scf_map.smeared_bin_density(true_density)
cr_density = (1.0 - scf_map.scf_fraction) * true_density
reco_density = cr_density + scf_density

true_mass = jnp.sum(true_density * areas)
reco_mass = jnp.sum(reco_density * areas)
print("true mass:", float(true_mass))
print("CR+SCF mass:", float(reco_mass))
print("relative difference:", float((reco_mass-true_mass)/true_mass))


## 3. Visualize true, migrated-SCF, and reconstructed densities


In [ ]:
def show(values, title):
    plt.figure(figsize=(6,5))
    plt.imshow(np.asarray(values).reshape(N_MP,N_TP).T, origin="lower", aspect="auto", extent=(0,1,0,1))
    plt.xlabel(r"$m'$"); plt.ylabel(r"$\\theta'$"); plt.title(title); plt.colorbar(); plt.show()

show(true_density, "True signal density")
show(scf_density, "Migrated SCF density")
show(reco_density, "CR + SCF reconstructed density")


## 4. Evaluate the reconstructed PDF on Dalitz points


In [ ]:
sample = model.generate_phase_space(20000, seed=7001)
values = pdf(sample.as_dict(), {})
print("normalization:", float(pdf.normalization({})))
fig, ax = plt.subplots(figsize=(7,5.5))
h = ax.hist2d(np.asarray(sample.s13), np.asarray(sample.s23), bins=70, weights=np.asarray(values))
fig.colorbar(h[3], ax=ax, label="PDF-weighted density")
ax.set(xlabel=r"$s_{13}$ [GeV$^2$]", ylabel=r"$s_{23}$ [GeV$^2$]", title="SCF-aware reconstructed signal")
plt.show()
